In [ ]:
from neuromaps import NeuroMapFixed

model_long = NeuroMapFixed.load("checkpoints/long/model.ckpt")
model_short = NeuroMapFixed.load("checkpoints/short/model.ckpt")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

hist_long = model_long.training_history
hist_short = model_short.training_history

plt.style.use("seaborn-v0_8")
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

# Строим графики истории обучения
if 'train_loss' in hist_long and len(hist_long['train_loss']) > 0:
    epochs_long = np.arange(len(hist_long['train_loss']))
    ax.plot(epochs_long, hist_long['train_loss'],
            label="Loss по траектории (fit_recursive)", lw=1.5, alpha=.8, color='blue')

if 'train_loss' in hist_short and len(hist_short['train_loss']) > 0:
    epochs_short = np.arange(len(hist_short['train_loss']))
    ax.plot(epochs_short, hist_short['train_loss'],
            label="Loss по одной точке (fit)", lw=1.5, alpha=.8, color='red')

ax.set_xlabel("Epoch")
ax.set_ylabel("Train loss")
ax.legend()
ax.grid(True, ls="--", alpha=.3)
plt.tight_layout()
plt.show()

In [ ]:
from utils import plot_compare_trajectories, get_attractor_trajectory
from systems import chua_rk4, chua_right_part

u0 = [ 2.98994669e+00, 4.49008896e-04, -4.52306348e+00]
p = [8.4, 12, 0, -0.12, -1.15]

# Симулируем траектории обеих моделей
nm_long_traj = model_long.simulate(u0=u0, p=p, n_steps=20000, divergence_threshold=1e3)
nm_short_traj = model_short.simulate(u0=u0, p=p, n_steps=20000, divergence_threshold=1e3)
ode_traj = get_attractor_trajectory(
    chua_rk4, chua_right_part, u0, p, 0.01, 50, 50, 
    lambda x, y: x[1], lambda x, y: [0, 1, 0], divergence_threshold=1e3
)

# Отбрасываем транзиентный процесс
if nm_long_traj is not None:
    nm_long_traj = nm_long_traj[10000:]
if nm_short_traj is not None:
    nm_short_traj = nm_short_traj[10000:]

plot_compare_trajectories(
    ode_traj, nm_long_traj, nm_short_traj, 
    labels=['ODE', 'Loss по траектории', 'Loss по одной точке'],
    layout='sidebyside'
)